<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_01_loss_functions_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 01 — Where the Loss Comes From

**Paired with L6.1 · Loss Functions and Gradients**

The recipe of L6.1: **write down the probability of the data given the model's
output, take the logarithm, put a minus sign in front. That is the loss.** You
will

1. derive the mean squared error from Gaussian noise, and check that the two
   have the same minimiser;
2. see what the noise level $\sigma$ does and does not change;
3. implement the cross entropy in NumPy and check it against
   `nn.CrossEntropyLoss`;
4. train one classifier under both losses and compare;
5. corrupt one reading and watch the fitted line move under each loss.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import Ex_6_core as core
core.keep_outputs()


In [ ]:
# A load cell, calibrated. Forty readings of a straight line plus noise.
#   core.calibration_dataset(n, seed, sigma) -> x, y, each (n,)
#   core.TRUE_SLOPE, core.TRUE_INTERCEPT, core.TRUE_SIGMA are what
#   generated them, so every fit below can be scored against the truth.
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_6_core as core

core.set_seed(0)

x, y = core.calibration_dataset(n=40)
print("calibration points:", x.shape)
print("true slope %.2f, intercept %.2f, noise sigma %.2f"
      % (core.TRUE_SLOPE, core.TRUE_INTERCEPT, core.TRUE_SIGMA))

fig, ax = plt.subplots(figsize=(6.6, 4.2))
ax.plot(x, core.TRUE_SLOPE * x + core.TRUE_INTERCEPT, lw=1.6, ls="--",
        color="#888888", label="true calibration")
ax.plot(x, y, "o", ms=6, color="#111111", label="readings")
ax.set_xlabel("applied load [normalised]"); ax.set_ylabel("sensor reading")
ax.set_title("Forty calibration points from a load cell")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25)
plt.show()

**What you should see.** Forty points scattered about a dashed line of slope 2.4
and intercept 0.8. The load cell's error is Gaussian, independent between
readings and of constant variance: exactly what the mean squared error assumes.

---

## 1 · From a Gaussian assumption to the mean squared error

This is not a fitting exercise. The question is *why least squares?* Write the
model as $\hat{y}(x) = a x + b$ and the assumption as

$$y_i = \hat{y}(x_i) + \varepsilon_i, \qquad
\varepsilon_i \sim \mathcal{N}(0, \sigma^2), \quad \text{independent.}$$

The density of one reading is

$$p(y_i \mid a, b) = \frac{1}{\sqrt{2\pi\sigma^2}}
\exp\!\left(-\frac{(y_i - \hat{y}(x_i))^2}{2\sigma^2}\right),$$

the density of the data is the product over $i$, and its logarithm with a minus
sign in front is the **negative log likelihood**

$$\mathcal{L}_{\mathrm{NLL}}(a, b) \;=\;
\underbrace{\frac{n}{2}\log(2\pi\sigma^2)}_{\text{does not contain } a, b}
\;+\; \frac{1}{2\sigma^2}\sum_{i=1}^{n}\bigl(y_i - \hat{y}(x_i)\bigr)^2 .$$

The first term does not contain the parameters and the second is the sum of
squared errors times a positive constant. So **minimising the Gaussian negative
log likelihood is minimising the mean squared error**, with the same minimiser.

### Your turn

Check it numerically: evaluate both objectives on a grid of $(a, b)$ and compare
where each is smallest.

In [ ]:
# two objectives, one minimiser ---------------------------------------------
def nll(a, b, sigma):
    residual = y - (a * x + b)
    n = len(x)
    return 0.5 * n * np.log(2 * np.pi * sigma ** 2) + np.sum(residual ** 2) / (2 * sigma ** 2)

def mse(a, b):
    return np.mean((y - (a * x + b)) ** 2)

a_grid = np.linspace(1.8, 3.0, 601)
b_grid = np.linspace(0.2, 1.4, 601)
NLL = np.array([[nll(a, b, core.TRUE_SIGMA) for b in b_grid] for a in a_grid])
MSE = np.array([[mse(a, b) for b in b_grid] for a in a_grid])

ia, ib = np.unravel_index(NLL.argmin(), NLL.shape)
a_nll, b_nll = a_grid[ia], b_grid[ib]
ia, ib = np.unravel_index(MSE.argmin(), MSE.shape)
a_mse, b_mse = a_grid[ia], b_grid[ib]
# ------------------------------------------------------------------------------

In [ ]:
# Least squares, solved directly.
#   np.stack([x, ones], axis=1) is the design matrix: one column for
#   the slope, one of ones for the intercept.
#   np.linalg.lstsq solves it in one step - no gradient descent, no
#   learning rate, and it is exact for a linear model under MSE.
# Two columns: the first multiplies the slope, the second - all
# ones - multiplies the intercept. That is how a bias becomes
# just another weight.
design = np.stack([x, np.ones_like(x)], axis=1)
a_ls, b_ls = np.linalg.lstsq(design, y, rcond=None)[0]

print(core.error_table(
    [["negative log likelihood (grid)", f"{a_nll:.4f}", f"{b_nll:.4f}"],
     ["mean squared error (grid)", f"{a_mse:.4f}", f"{b_mse:.4f}"],
     ["least squares (exact)", f"{a_ls:.4f}", f"{b_ls:.4f}"],
     ["the truth", f"{core.TRUE_SLOPE:.4f}", f"{core.TRUE_INTERCEPT:.4f}"]],
    ["objective", "slope", "intercept"]))

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.4))
for ax, G, name in zip(axes, [NLL, MSE],
                       ["negative log likelihood", "mean squared error"]):
    cs = ax.contour(b_grid, a_grid, G, levels=25, cmap="viridis")
    ax.plot(b_ls, a_ls, "x", ms=13, mew=2.4, color="#d94f2b",
            label="least squares")
    ax.plot(core.TRUE_INTERCEPT, core.TRUE_SLOPE, "o", ms=9, mfc="none",
            mec="#111111", mew=2.0, label="truth")
    ax.set_xlabel("intercept $b$"); ax.set_ylabel("slope $a$")
    ax.set_title(name); ax.legend(frameon=False, fontsize=9)
plt.show()

**What you should see.**

| objective | slope | intercept |
| --- | --- | --- |
| negative log likelihood (grid) | 2.3960 | 0.7860 |
| mean squared error (grid) | 2.3960 | 0.7860 |
| least squares (exact) | 2.3954 | 0.7871 |
| the truth | 2.4000 | 0.8000 |

and two contour plots with identical ellipses and identical centres.

The first two rows agree exactly; their gap to the exact least squares is the
grid spacing, 0.002. The gap to the truth is **estimation error**: forty noisy
readings do not pin down the true line, and only more data fixes that.

---

## 2 · What $\sigma$ changes, and what it does not

$\sigma$ multiplies the sum of squares by a constant and adds another constant,
so it does not move the minimiser. It still does two things.

In [ ]:
# The same fit read as a likelihood: with Gaussian noise, minimising
# the squared error IS maximising the likelihood, and the assumed
# sigma changes the number but not the minimiser.
print("negative log likelihood at the fitted line, for several assumed sigma:")
for s in (0.1, 0.2, core.TRUE_SIGMA, 0.5, 1.0):
    print(f"   sigma = {s:.2f}   NLL = {nll(a_ls, b_ls, s):9.4f}")

residual = y - (a_ls * x + b_ls)
sigma_hat = float(np.sqrt(np.mean(residual ** 2)))
print(f"\nsigma estimated from the residuals: {sigma_hat:.4f}"
      f"   (the instrument's true value: {core.TRUE_SIGMA:.4f})")

fig, ax = plt.subplots(1, 2, figsize=(12.0, 3.4))
ax[0].plot(x, residual, "o", ms=5, color="#1f77b4")
ax[0].axhline(0.0, color="#111111", lw=1.0)
ax[0].set_xlabel("applied load"); ax[0].set_ylabel("residual")
ax[0].set_title("residuals against load — should be structureless")
ax[1].hist(residual, bins=12, color="#1f77b4", alpha=0.85)
ax[1].set_xlabel("residual"); ax[1].set_title("residual histogram")
for a in ax:
    a.grid(alpha=0.25)
plt.show()

**What you should see.** Negative log likelihoods of about 186.6, 32.9, 14.5,
18.7 and 39.2 for the five values of $\sigma$, an estimated $\sigma$ of about
0.3478 against a true 0.35, and residual plots with no obvious structure.

**$\sigma$ sets the value of the loss**, so it is needed to compare two models
by their likelihood, though not to fit one.

**$\sigma$ can be estimated.** Minimising the negative log likelihood over
$\sigma$ as well gives $\hat{\sigma}^2 = \frac{1}{n}\sum r_i^2$, the printed
estimate. That it lands near 0.35 says the model is adequate; at 2.0 it would be
missing something real. A network that predicts $\sigma$ as well as $\mu$, as in
L6.1, makes this estimate for every input.

---

## 3 · From a categorical assumption to the cross entropy

The output is now one of $C$ classes. The network outputs **logits** $z_c$, and
the softmax turns them into probabilities,

$$p_c = \frac{e^{z_c}}{\sum_{k} e^{z_k}}.$$

The recipe gives minus the log of the probability of the observed label $t$,

$$\mathcal{L} = -\log p_t
= -z_t + \log\sum_k e^{z_k},$$

and averaged over the data that is the **cross entropy**.

Two details. **Subtract the largest logit before exponentiating**: $e^{z}$
overflows above about 88 in single precision, and the softmax does not change.
**`nn.CrossEntropyLoss` takes logits, not probabilities**: apply a softmax first
and it is applied twice, which trains slowly to a worse answer with no warning.

### Your turn

Implement the softmax and the cross entropy in NumPy, and check against PyTorch.

In [ ]:
# softmax and cross entropy, stably -----------------------------------------
def softmax_np(z):                 # z has shape (N, C)
    shifted = z - z.max(axis=1, keepdims=True)    # subtract the row max: no overflow in exp
    e = np.exp(shifted)
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy_np(z, t):        # t has shape (N,), integer labels
    shifted = z - z.max(axis=1, keepdims=True)
    log_p   = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    return -log_p[np.arange(len(t)), t].mean()

logits = np.array([[2.0, 1.0, 0.1],
                   [0.5, 2.5, 0.3],
                   [1.2, 0.7, 3.1],
                   [0.0, 0.0, 0.0],
                   [8.0, -2.0, -3.0],
                   [-1.0, -1.0, 9.0]])
labels = np.array([0, 1, 2, 1, 2, 2])

ce_numpy = float(cross_entropy_np(logits, labels))
ce_torch = float(nn.CrossEntropyLoss()(torch.tensor(logits), torch.tensor(labels)))
# ------------------------------------------------------------------------------

In [ ]:
# softmax turns scores into probabilities; cross entropy scores the
# probability given to the correct class. Both written by hand here,
# in the numerically stable form: subtract the row maximum before
# the exponential, or a large logit overflows.
probs = softmax_np(logits)
print("softmax rows sum to one:", np.allclose(probs.sum(axis=1), 1.0))
print("probabilities:")
for row, t in zip(probs, labels):
    print("   " + "  ".join(f"{p:.4f}" for p in row) + f"    label {t}")
print()
print(f"cross entropy, NumPy : {ce_numpy:.10f}")
print(f"cross entropy, torch : {ce_torch:.10f}")
print(f"difference           : {abs(ce_numpy - ce_torch):.2e}")
print()
print("loss for a uniform three-class guess: ln 3 =", round(float(np.log(3)), 6))

**What you should see.** `softmax rows sum to one: True`, a table of
probabilities, and

```
cross entropy, NumPy : 2.1585311972
cross entropy, torch : 2.1585311972
difference           : 0.00e+00
```

or a difference around $10^{-16}$.

Row four has all logits zero, so every probability is one third and the loss is
$\ln 3 = 1.0986$, the number printed last. A three-class cross entropy that does
not start near 1.10 means the labels, the shapes or the initialisation are
wrong; for two classes it is $\ln 2 = 0.693$, for ten $\ln 10 = 2.303$. Row five,
logits `[8, -2, -3]` with label 2, is confidently wrong and costs about 11.

---

## 4 · The two losses on the same problem

Train the same small network on the three vibration classes twice: with cross
entropy on the logits, and with mean squared error on the softmax outputs
against one-hot targets, which is what many people write when they come to
classification from regression.

### Your turn

In [ ]:
# the same network under two losses -----------------------------------------
X, y_cls = core.vibration_dataset()
X_train, y_train = X[:240], y_cls[:240]
X_test,  y_test  = X[240:], y_cls[240:]

Xa = torch.tensor(X_train); ya = torch.tensor(y_train)
Xb = torch.tensor(X_test)
Y_onehot = torch.tensor(core.one_hot(y_train).astype(np.float32))

def loss_ce(model):
    return nn.CrossEntropyLoss()(model(Xa), ya)

def loss_mse(model):
    return nn.MSELoss()(torch.softmax(model(Xa), dim=-1), Y_onehot)

results, models_cls = {}, {}
for name, loss_of in (("cross entropy", loss_ce), ("mean squared error", loss_mse)):
    core.set_seed(0)
    model = nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 3))
    optimiser = torch.optim.Adam(model.parameters(), lr=0.05)
    for epoch in range(400):
        optimiser.zero_grad()
        loss_of(model).backward()
        optimiser.step()
    with torch.no_grad():
        logits_test = model(Xb)
        acc  = float((logits_test.argmax(dim=1).numpy() == y_test).mean())
        held = float(nn.CrossEntropyLoss()(logits_test, torch.tensor(y_test)))   # the same yardstick for both
    results[name] = (acc, held)
    models_cls[name] = model
# ------------------------------------------------------------------------------

In [ ]:
# The same network trained under both losses. Accuracy is close; the
# gradients are not, which is the point of the next cell.
print(core.error_table(
    [[name, f"{acc:.3f}", f"{held:.4f}"] for name, (acc, held) in results.items()],
    ["training loss", "held-out accuracy", "held-out cross entropy"]))

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.6))
for ax, name in zip(axes, results):
    with torch.no_grad():
        pred = models_cls[name](Xb).numpy().argmax(axis=1)
    core.plot_classes(X_test, y_test, ax=ax, predictions=pred,
                      title=f"trained with {name}")
plt.show()

**What you should see.**

| training loss | held-out accuracy | held-out cross entropy |
| --- | --- | --- |
| cross entropy | 0.967 | 0.1191 |
| mean squared error | 0.967 | 0.1687 |

**The accuracies are identical**: on well-separated clusters both losses find
the same decision boundary. **The held-out cross entropies are not**: the model
trained with squared error is about forty per cent worse at the probabilities,
so its confidence is a worse guide to its accuracy. For an alarm set at
$p(\text{bearing fault}) > 0.8$, that difference is the whole system. It comes
from the gradient.

---

## 5 · The gradient when the model is confidently wrong

Two classes, one logit $z$, $p = \sigma(z)$, and a sample whose true label is 1.

**Cross entropy.** $\mathcal{L} = -\log p$, and
$$\frac{\partial \mathcal{L}}{\partial z} = p - 1.$$

**Squared error on the probability.** $\mathcal{L} = (p - 1)^2$, and with
$\sigma' = p(1-p)$,
$$\frac{\partial \mathcal{L}}{\partial z} = 2(p - 1)\,p\,(1 - p).$$

The second has an extra factor $p(1-p)$, which goes to zero at both ends.

In [ ]:
# Why cross entropy is used for classification: at a confidently
# wrong logit, the MSE gradient has gone flat and the cross-entropy
# gradient has not, so only one of them still learns.
print("  logit z    p        dCE/dz      dMSE/dz")
for z in (-6.0, -3.0, 0.0, 3.0, 6.0):
    p = 1.0 / (1.0 + np.exp(-z))
    g_ce = p - 1.0
    g_mse = 2.0 * (p - 1.0) * p * (1.0 - p)
    print(f"  {z:6.1f}   {p:.4f}   {g_ce: .4f}   {g_mse: .6f}")

zs = np.linspace(-8, 8, 400)
ps = 1.0 / (1.0 + np.exp(-zs))
fig, ax = plt.subplots(figsize=(6.8, 4.2))
ax.plot(zs, np.abs(ps - 1.0), lw=1.9, color="#d94f2b", label="cross entropy")
ax.plot(zs, np.abs(2 * (ps - 1) * ps * (1 - ps)), lw=1.9, color="#1f77b4",
        label="squared error on the probability")
ax.set_yscale("log")
ax.set_xlabel("logit $z$   (true label is 1, so $z \\ll 0$ is confidently wrong)")
ax.set_ylabel("|gradient with respect to $z$|")
ax.set_title("What each loss does about a confident mistake")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

**What you should see.**

```
  logit z    p        dCE/dz      dMSE/dz
    -6.0   0.0025   -0.9975   -0.004921
    -3.0   0.0474   -0.9526   -0.086068
     0.0   0.5000   -0.5000   -0.250000
     3.0   0.9526   -0.0474   -0.004285
     6.0   0.9975   -0.0025   -0.000012
```

and a log-scale figure in which the red curve stays high on the left while the
blue one collapses.

In the first row the model gives the correct class a probability of 0.0025.
Cross entropy answers with a gradient of $-0.9975$, squared error with
$-0.0049$, **two hundred times smaller**: the sample the model most needs to
learn from is the one it learns least from. That is why cross entropy is the
standard choice. On the right, where the model is confidently right, both
gradients are small, as they should be.

---

## 6 · Change the noise assumption, change the loss

A bad reading, a stuck bit or a knocked cable, is not Gaussian. A Gaussian gives
a residual of $10\sigma$ a probability of about $10^{-23}$, so squared error
moves the whole line to accommodate it. Assume Laplace noise instead,
$p(r) \propto e^{-|r|/b}$, where large errors are less rare, and the recipe gives
$-\log p \propto |r|$: the **mean absolute error**, which weights a large
residual only linearly.

### Your turn

Corrupt one reading and fit the same line under both losses.

In [ ]:
# one bad reading, two losses -----------------------------------------------
x_bad, y_bad = core.add_outlier(x, y, index=7, offset=4.0)
xb, yb = core.to_tensor(x_bad), core.to_tensor(y_bad)

fits = {}
for name, loss_fn in (("MSE", nn.MSELoss()), ("MAE", nn.L1Loss())):
    core.set_seed(0)
    line = nn.Linear(1, 1)
    optimiser = torch.optim.Adam(line.parameters(), lr=0.05)
    for epoch in range(3000):
        optimiser.zero_grad()
        loss = loss_fn(line(xb), yb)
        loss.backward()
        optimiser.step()
    fits[name] = (float(line.weight.item()), float(line.bias.item()))
# ------------------------------------------------------------------------------

In [ ]:
# One reading recorded wrongly (core.add_outlier) and two fits: the
# squared error chases the outlier, the absolute error mostly does not.
print(core.error_table(
    [[name, f"{a:.4f}", f"{b:.4f}",
      f"{abs(a - core.TRUE_SLOPE):.4f}", f"{abs(b - core.TRUE_INTERCEPT):.4f}"]
     for name, (a, b) in fits.items()]
    + [["truth", f"{core.TRUE_SLOPE:.4f}", f"{core.TRUE_INTERCEPT:.4f}",
        "-", "-"]],
    ["loss", "slope", "intercept", "slope error", "intercept error"]))

grid = np.linspace(0, 2, 100)
fig, ax = plt.subplots(figsize=(7.0, 4.4))
ax.plot(grid, core.TRUE_SLOPE * grid + core.TRUE_INTERCEPT, lw=1.6, ls="--",
        color="#888888", label="truth")
for i, (name, (a, b)) in enumerate(fits.items()):
    ax.plot(grid, a * grid + b, lw=2.0,
            color=["#d94f2b", "#1f77b4"][i], label=f"fitted with {name}")
ax.plot(x_bad, y_bad, "o", ms=6, color="#111111", label="readings")
ax.plot(x_bad[7], y_bad[7], "o", ms=13, mfc="none", mec="#d94f2b", mew=2.4,
        label="the bad reading")
ax.set_xlabel("applied load"); ax.set_ylabel("sensor reading")
ax.set_title("One bad reading in forty")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25)
plt.show()

**What you should see.**

| loss | slope | intercept | slope error | intercept error |
| --- | --- | --- | --- | --- |
| MSE | 2.1824 | 1.0762 | 0.2176 | 0.2762 |
| MAE | 2.3616 | 0.8086 | 0.0384 | 0.0086 |
| truth | 2.4000 | 0.8000 | - | - |

and a figure in which the red MSE line is dragged towards the circled point
while the blue MAE line ignores it.

One bad reading in forty moved the least-squares slope by nine per cent and the
intercept by thirty-five; the absolute-error fit stays within two per cent of
the truth. The absolute error is not free: it is not differentiable at zero,
converges more slowly, and wastes information when the noise really is
Gaussian. The **Huber** loss, `nn.SmoothL1Loss`, is quadratic for small
residuals and linear for large ones, and is what you would use on instrument
data.

---

## 7 · Save

In [ ]:
# Saved for the report in notebook 06.
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb01_losses.npz")
np.savez(path,
         a_nll=a_nll, b_nll=b_nll, a_mse=a_mse, b_mse=b_mse,
         a_ls=a_ls, b_ls=b_ls, sigma_hat=sigma_hat,
         ce_numpy=ce_numpy, ce_torch=ce_torch,
         acc_ce=results["cross entropy"][0],
         acc_mse=results["mean squared error"][0],
         heldout_ce_ce=results["cross entropy"][1],
         heldout_ce_mse=results["mean squared error"][1],
         fit_mse=np.asarray(fits["MSE"]), fit_mae=np.asarray(fits["MAE"]))
print("wrote", path)
core.saved(path)


**What you should see.** `wrote .../Ex06_outputs/nb01_losses.npz`.

---

## 8 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. The negative log likelihood and the mean squared error had the same
   minimiser. Using this as the example, say where a loss function comes from —
   the steps from choosing a distribution to minimising the negative log
   likelihood — and give one thing you can do with the first that you cannot do
   with the second.
   *→ L6.1 Q1, Q2*
2. Your fitted $\hat{\sigma}$ came out near the instrument's true value. What
   distribution does squared error assume? Suppose $\hat{\sigma}$ had come out
   four times larger: name two different explanations, and say how you would
   tell them apart.
   *→ L6.1 Q3*
3. Both losses gave the same accuracy on the vibration data but different
   held-out cross entropies. Where does cross entropy come from, and what should
   an untrained model report here, with three classes — and with ten? Describe an
   application in which you would not care about the difference, and one in
   which you would refuse to deploy the worse-calibrated model.
   *→ L6.1 Q5*
4. You are given a dataset in which about one reading in fifty is a
   transcription error, and the rest are Gaussian. Write down the loss you would
   use and the assumption it corresponds to, and say what it costs you if the
   noise was Gaussian after all.
   *→ L6.1 Q4*

*Write your answers here. You will copy them into the report in notebook 06, which adds them to what you submit.*

1.
2.
3.
4.

---

Next: **[notebook 02](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_02_optimiser_comparison_light.ipynb)**, where this same loss is minimised four different
ways — and where the Adam-then-L-BFGS recipe that every exercise in Part 2
uses is measured rather than asserted.
